# `doe-optimization-toolkit`: Design of Experiments & Response Surfaces

Programmatic industrial engineering experimental design and surface modeling:
1. **Full & Fractional Factorials**: $2^k, 3^k, 2^{k-p}$, defining relations, and alias structures.
2. **Diagnostic Tools**: Main effects, 2-factor interactions, and Daniel's half-normal probability plots.
3. **Response Surface Methodology (RSM)**: Central Composite Design (CCD), Box-Behnken (BBD), and quadratic stationary point optimization.

In [1]:
from doe_toolkit.factorial import full_factorial_2k, fractional_factorial_2k_p
from doe_toolkit.diagnostics import calculate_effects, half_normal_plot_coordinates
from doe_toolkit.rsm import central_composite_design, QuadraticResponseSurface

## 1. Full 2^k Factorials with Natural Engineering Scaling

In [2]:
factors = ["Temperature", "Pressure", "Catalyst"]
levels = {"Temperature": (120.0, 180.0), "Pressure": (2.0, 6.0), "Catalyst": (0.5, 2.5)}
full_design = full_factorial_2k(factors, center_points=2, factor_levels=levels)
print(full_design.to_natural_dataframe())

## 2. Fractional Factorials 2^(k-p) with Alias Structure Analysis

In [3]:
factors_4 = ["A", "B", "C", "D"]
generators = {"D": "A*B*C"}
frac_design, alias_map, resolution = fractional_factorial_2k_p(factors_4, generators)
print(f"Resolution: {resolution}")
print(f"Defining Relation: {frac_design.extra_info['defining_relation']}")
for eff, al in list(alias_map.items())[:4]:
    print(f"{eff} = {' + '.join(al)}")

## 3. Central Composite Design & Response Surface Optimization

In [4]:
rsm_factors = ["Temp", "Conc"]
rsm_levels = {"Temp": (140.0, 180.0), "Conc": (10.0, 30.0)}
ccd = central_composite_design(rsm_factors, alpha="rotatable", center_points=5, factor_levels=rsm_levels)
X_ccd = ccd.coded_matrix
y_rsm = 92.0 - 6.0 * (X_ccd[:, 0] - 0.3) ** 2 - 8.0 * (X_ccd[:, 1] + 0.4) ** 2

surface = QuadraticResponseSurface().fit(ccd, y_rsm)
res = surface.optimize()
print(f"Stationary Point Nature: {res.nature}")
print(f"Optimal Natural Units: {res.point_natural}")
print(f"Optimal Predicted Response: {res.predicted_response:.2f}%")